# Matplotlib Styling and Readability

This notebook is all about making charts clear, consistent, and presentation-ready.


**Learning objectives**
- Improve titles, labels, and legends
- Format dates cleanly
- Build subplot grids for comparisons
- Use color and line styles intentionally

**Estimated time:** 45-60 minutes


## Setup


In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if (BASE_DIR / "viz-workshop").exists():
    BASE_DIR = BASE_DIR / "viz-workshop"
elif (BASE_DIR / "notebooks").exists():
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR / "../../src"))

from data_prep import load_or_create_dataset, get_paths, print_environment_hint

print_environment_hint()
paths = get_paths(BASE_DIR)

In [ ]:
# Load or build the processed dataset (cached after first run)
df = load_or_create_dataset(base_dir=BASE_DIR)
print(df.shape)
df.head()

## Episode 1: Labels, legends, annotation

Small labels can make a big difference for comprehension.


In [ ]:
import matplotlib.pyplot as plt
from viz_helpers import apply_workshop_style, quick_title

apply_workshop_style()

member_counts = df["member_casual"].value_counts()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(member_counts.index, member_counts.values, color=["#1f77b4", "#ff7f0e"])
quick_title(ax, "Member vs Casual Trips", "A quick breakdown of rider types")
ax.set_xlabel("Rider type")
ax.set_ylabel("Trips")
plt.tight_layout()
plt.show()

## Episode 2: Date formatting

Dates get messy fast. Rotating ticks helps.


In [ ]:
trips_by_day = df.groupby("date").size().reset_index(name="trips")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(trips_by_day["date"], trips_by_day["trips"], color="#2ca02c")
ax.set_title("Daily Trips (clean ticks)")
ax.set_xlabel("Date")
ax.set_ylabel("Trips")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Episode 3: Subplots and small multiples

Compare member vs casual side-by-side.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

for ax, rider_type in zip(axes, ["member", "casual"]):
    subset = df[df["member_casual"] == rider_type]
    by_hour = subset.groupby("hour").size()
    ax.plot(by_hour.index, by_hour.values, color="#1f77b4" if rider_type == "member" else "#ff7f0e")
    ax.set_title(rider_type.title())
    ax.set_xlabel("Hour")

axes[0].set_ylabel("Trips")
plt.tight_layout()
plt.show()

## Episode 4: Color, alpha, linewidth rules of thumb

- Use stronger contrast for key series
- Use alpha for dense plots
- Keep line widths consistent across a chart


## Episode 5: Handling outliers

Durations can be long. We already clipped to 6 hours in preprocessing. When in doubt, say so in the chart.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(df["duration_min"], bins=50, color="#8c564b")
ax.set_title("Trip Duration (clipped at 6 hours)")
ax.set_xlabel("Duration (min)")
ax.set_ylabel("Trips")
plt.tight_layout()
plt.show()

## Episode 6: Consistent style helper

We can reuse `apply_workshop_style()` to keep charts consistent.


## Episode 7: Export formats

Use PNG for slides and SVG for crisp web graphics.


In [ ]:
from pathlib import Path

outputs_dir = BASE_DIR / "outputs"
outputs_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(df["duration_min"], bins=40, color="#2ca02c")
ax.set_title("Trip Duration (exported)")

fig.savefig(outputs_dir / "duration_hist.png", dpi=150, bbox_inches="tight")
fig.savefig(outputs_dir / "duration_hist.svg", bbox_inches="tight")
plt.show()

## Exercise 1: Ugly to presentation-ready
Take a quick plot and apply the checklist: title, labels, readable ticks, and consistent color.

**Check yourself:** the updated chart should look noticeably cleaner.


## Exercise 2: 2x2 panel
Create a 2x2 subplot grid with shared axes and consistent scales.
Try: member vs casual, weekday vs weekend, or rideable type splits.


**Solution (optional): 2x2 panel**

In [ ]:
import numpy as np

is_weekend = df["day_of_week"] >= 5

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

panels = [
    (df[(df["member_casual"] == "member") & (~is_weekend)], "Member - Weekday"),
    (df[(df["member_casual"] == "member") & (is_weekend)], "Member - Weekend"),
    (df[(df["member_casual"] == "casual") & (~is_weekend)], "Casual - Weekday"),
    (df[(df["member_casual"] == "casual") & (is_weekend)], "Casual - Weekend"),
]

for ax, (subset, title) in zip(axes.ravel(), panels):
    by_hour = subset.groupby("hour").size()
    ax.plot(by_hour.index, by_hour.values)
    ax.set_title(title)

plt.tight_layout()
plt.show()

## Common pitfalls
- Rotating date labels: it helps readability, but avoid overcrowding by limiting ticks.
- Too many colors: pick one highlight color and keep others neutral.


## Wrap-up
We now have a consistent Matplotlib style. Next, we move to Seaborn and Plotly for higher-level and interactive charts.

**What’s next:** `03_seaborn_plotly.ipynb`
